# Multi-omics (RNA-seq + Proteomics) — Live-coding Notebook Template
This notebook is designed for step-by-step execution in a live coding setting.

**Files expected**:
- `data/metadata.csv`
- `data/transcriptomics.csv`
- `data/proteomics.csv`

**Metadata columns expected**: `sample_id, subject_id, treatment, timepoint` (optional: `batch`).

Targets: `GENE001`, `PROT001` (edit in the config cell).

In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from scipy import stats
from statsmodels.stats.multitest import multipletests

# --- Config
DATA_DIR = "data"
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

META_PATH = os.path.join(DATA_DIR, "metadata.csv")
RNA_PATH  = os.path.join(DATA_DIR, "transcriptomics.csv")
PROT_PATH = os.path.join(DATA_DIR, "proteomics.csv")

BASELINE = "baseline"
WEEK4 = "week4"

GENE_TARGET = "GENE001"
PROT_TARGET = "PROT001"

# Thresholds (edit if needed)
FDR_THR = 0.05
LFC_THR_RNA = 0.5
LFC_THR_PROT = 0.25

np.random.seed(0)

print("Working dir:", os.getcwd())
print("Data dir exists:", os.path.isdir(DATA_DIR))
if os.path.isdir(DATA_DIR):
    print("Data files:", os.listdir(DATA_DIR))
print("Outputs dir:", OUT_DIR)


In [ ]:
# --- Load raw files + quick sanity
meta_raw = pd.read_csv(META_PATH)
rna_raw_file = pd.read_csv(RNA_PATH)
prot_raw_file = pd.read_csv(PROT_PATH)

print("meta_raw:", meta_raw.shape)
print("rna_raw_file:", rna_raw_file.shape)
print("prot_raw_file:", prot_raw_file.shape)

display(meta_raw.head())
display(rna_raw_file.head(3))
display(prot_raw_file.head(3))


In [ ]:
# --- Helper: flexible matrix loader (returns samples x features)

def load_matrix_flexible_csv(df: pd.DataFrame) -> pd.DataFrame:
    """Return samples x features with index=sample_id."""
    df = df.copy()

    # Case: has sample_id column already -> sample x feature
    for c in df.columns:
        if c.lower() in ("sample_id", "sample", "sid"):
            df = df.set_index(c)
            df.index.name = "sample_id"
            return df.apply(pd.to_numeric, errors="coerce")

    # Case: first column is feature IDs, rest are samples -> transpose
    first_col = df.columns[0]
    feat_by_sample = df.set_index(first_col)
    sample_by_feat = feat_by_sample.T
    sample_by_feat.index.name = "sample_id"
    return sample_by_feat.apply(pd.to_numeric, errors="coerce")


rna = load_matrix_flexible_csv(rna_raw_file)     # samples x genes
prot = load_matrix_flexible_csv(prot_raw_file)   # samples x proteins

print("RNA matrix (samples x genes):", rna.shape)
print("Proteomics matrix (samples x proteins):", prot.shape)
print("RNA sample ids example:", list(rna.index[:5]))
print("RNA genes example:", list(rna.columns[:5]))
print("Prot proteins example:", list(prot.columns[:5]))


In [ ]:
# --- Clean metadata + align to matrices + pairing QC

def normalize_treatment(x: str) -> str:
    x0 = str(x).strip().lower()
    return "placebo" if "plac" in x0 else "ASO"

def normalize_timepoint(x: str) -> str:
    x0 = str(x).strip().lower()
    if x0 in ("baseline", "base", "week0", "wk0", "0", "t0"):
        return BASELINE
    if x0 in ("week4", "wk4", "4", "t4"):
        return WEEK4
    return str(x).strip()

meta = meta_raw.copy()
meta["sample_id"] = meta["sample_id"].astype(str).str.strip()
meta["subject_id"] = meta["subject_id"].astype(str).str.strip()
meta["treatment"] = meta["treatment"].map(normalize_treatment)
meta["timepoint"] = meta["timepoint"].map(normalize_timepoint)

# Align
meta_rna = meta[meta["sample_id"].isin(rna.index)].copy()
rna = rna.loc[meta_rna["sample_id"]]

meta_prot = meta[meta["sample_id"].isin(prot.index)].copy()
prot = prot.loc[meta_prot["sample_id"]]

print("Aligned RNA:", meta_rna.shape, rna.shape)
print("Aligned Prot:", meta_prot.shape, prot.shape)

# QC tables
print("\nQC RNA counts:")
display(pd.crosstab(meta_rna["treatment"], meta_rna["timepoint"]))

print("\nQC Prot counts:")
display(pd.crosstab(meta_prot["treatment"], meta_prot["timepoint"]))

# Pairing
pair = pd.crosstab(meta["subject_id"], meta["timepoint"])
paired_subjects = pair[(pair.get(BASELINE,0)>0) & (pair.get(WEEK4,0)>0)].index
print("\nPaired subjects:", len(paired_subjects), "/", pair.shape[0])


In [ ]:
# --- Transforms (QC only) + PCA plots (treatment/timepoint/batch)

# RNA log2CPM (QC/visualization)
rna_counts = rna.fillna(0)
rna_counts = rna_counts.clip(lower=0).astype(int)

lib = rna_counts.sum(axis=1).replace(0, np.nan)
rna_logcpm = np.log2((rna_counts.div(lib, axis=0)*1e6) + 1)

# Proteomics log2
prot_log2 = np.log2(prot.replace(0, np.nan))

print("RNA log2CPM:", rna_logcpm.shape, "min/max:", np.nanmin(rna_logcpm.values), np.nanmax(rna_logcpm.values))
print("Prot log2:", prot_log2.shape)

def pca_plot(mat: pd.DataFrame, meta_df: pd.DataFrame, color_col: str, title: str):
    mat2 = mat.loc[:, mat.notna().all(axis=0)]
    if mat2.shape[1] < 2:
        print("Too few complete features for PCA.")
        return
    X = StandardScaler().fit_transform(mat2.values)
    pcs = PCA(2, random_state=0).fit_transform(X)

    df = meta_df.copy()
    df["PC1"], df["PC2"] = pcs[:,0], pcs[:,1]

    plt.figure(figsize=(6,4))
    for v in df[color_col].astype(str).unique():
        sub = df[df[color_col].astype(str)==v]
        plt.scatter(sub.PC1, sub.PC2, alpha=0.75, label=f"{color_col}={v}")
    plt.title(title)
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

pca_plot(rna_logcpm, meta_rna, "treatment", "RNA PCA (colored by treatment)")
pca_plot(rna_logcpm, meta_rna, "timepoint", "RNA PCA (colored by timepoint)")
if "batch" in meta_rna.columns:
    pca_plot(rna_logcpm, meta_rna, "batch", "RNA PCA (colored by batch)")

pca_plot(prot_log2, meta_prot, "treatment", "Proteomics PCA (colored by treatment)")
if "batch" in meta_prot.columns:
    pca_plot(prot_log2, meta_prot, "batch", "Proteomics PCA (colored by batch)")


In [ ]:
# --- Target plots (GENE001 + PROT001) + Δ comparison

def long_feature(mat: pd.DataFrame, meta_df: pd.DataFrame, feature: str) -> pd.DataFrame:
    if feature not in mat.columns:
        raise KeyError(f"{feature} not found")
    df = meta_df[["sample_id","subject_id","treatment","timepoint"]].copy()
    df["value"] = pd.to_numeric(mat[feature].values, errors="coerce")
    return df

def box_jitter(df: pd.DataFrame, title: str):
    df = df.copy()
    df["group"] = df["treatment"].astype(str) + " / " + df["timepoint"].astype(str)
    groups = sorted(df["group"].unique())
    data = [df.loc[df["group"]==g, "value"].dropna().values for g in groups]

    plt.figure(figsize=(9,4))
    plt.boxplot(data, labels=groups, showfliers=False)
    for i, vals in enumerate(data, start=1):
        x = np.random.normal(i, 0.05, size=len(vals))
        plt.scatter(x, vals, alpha=0.6)
    plt.xticks(rotation=25, ha="right")
    plt.title(title)
    plt.tight_layout()
    plt.show()

def subject_delta(df: pd.DataFrame) -> pd.DataFrame:
    wide = df.pivot_table(index=["subject_id","treatment"], columns="timepoint", values="value", aggfunc="mean").reset_index()
    wide["delta"] = wide.get(WEEK4, np.nan) - wide.get(BASELINE, np.nan)
    return wide

def compare_delta(delta_df: pd.DataFrame, label: str):
    print("\nDelta summary:", label)
    display(delta_df.groupby("treatment")["delta"].agg(["count","mean","std"]))
    a = delta_df.loc[delta_df["treatment"].astype(str)=="ASO", "delta"].dropna().values
    p = delta_df.loc[delta_df["treatment"].astype(str)=="placebo", "delta"].dropna().values
    if len(a)>=2 and len(p)>=2:
        print("Welch t-test p=", stats.ttest_ind(a, p, equal_var=False).pvalue)
        print("Mann–Whitney p=", stats.mannwhitneyu(a, p, alternative="two-sided").pvalue)

# RNA target
gene_df = long_feature(rna_logcpm, meta_rna, GENE_TARGET)
box_jitter(gene_df, f"{GENE_TARGET} RNA (log2CPM)")
gene_delta = subject_delta(gene_df)
compare_delta(gene_delta, f"{GENE_TARGET} RNA Δ(week4-baseline)")

# Proteomics target
prot_df = long_feature(prot_log2, meta_prot, PROT_TARGET)
box_jitter(prot_df, f"{PROT_TARGET} Protein (log2 intensity)")
prot_delta = subject_delta(prot_df)
compare_delta(prot_delta, f"{PROT_TARGET} Protein Δ(week4-baseline)")


In [ ]:
# --- Correlation filtering: Δtarget vs Δall (RNA + proteomics)

def delta_matrix(meta_df: pd.DataFrame, mat: pd.DataFrame) -> pd.DataFrame:
    df = meta_df[["sample_id","subject_id","timepoint"]].set_index("sample_id").join(mat, how="inner")
    base = df[df.timepoint==BASELINE].groupby("subject_id").mean(numeric_only=True)
    wk4  = df[df.timepoint==WEEK4].groupby("subject_id").mean(numeric_only=True)
    common = base.index.intersection(wk4.index)
    return (wk4.loc[common] - base.loc[common])

def correlate_target(delta: pd.DataFrame, target: str, min_n=8):
    y = pd.to_numeric(delta[target], errors="coerce")
    rows=[]
    for feat in delta.columns:
        if feat == target: 
            continue
        x = pd.to_numeric(delta[feat], errors="coerce")
        m = x.notna() & y.notna()
        if m.sum() < min_n:
            continue
        r, p = stats.spearmanr(x[m], y[m])
        rows.append([feat, int(m.sum()), r, p])
    res = pd.DataFrame(rows, columns=["feature","n","corr","pvalue"])
    res["qvalue"] = multipletests(res["pvalue"].values, method="fdr_bh")[1]
    return res.sort_values(["qvalue","pvalue"])

delta_rna_all = delta_matrix(meta_rna, rna_logcpm)
delta_prot_all = delta_matrix(meta_prot, prot_log2)

rna_corr = correlate_target(delta_rna_all, GENE_TARGET)
prot_corr = correlate_target(delta_prot_all, PROT_TARGET)

display(rna_corr.head(10))
display(prot_corr.head(10))

rna_corr.to_csv(f"{OUT_DIR}/corr_delta_{GENE_TARGET}_RNA.csv", index=False)
prot_corr.to_csv(f"{OUT_DIR}/corr_delta_{PROT_TARGET}_PROT.csv", index=False)
print("Saved correlation tables to outputs/")


In [ ]:
# --- DE (delta-based) + ASO/Placebo up/down sets + Venn overlap figures

try:
    from matplotlib_venn import venn2
    HAS_VENN = True
except Exception as e:
    print("matplotlib_venn not available; will print overlap counts instead.", e)
    HAS_VENN = False

def de_from_deltas_by_arm(meta_df: pd.DataFrame, delta: pd.DataFrame, arm: str, min_n=6):
    subs = meta_df.loc[meta_df["treatment"].astype(str)==arm, "subject_id"].unique()
    d = delta.loc[delta.index.isin(subs)]

    rows=[]
    for feat in d.columns:
        v = pd.to_numeric(d[feat], errors="coerce").dropna()
        if len(v) < min_n:
            continue
        t = stats.ttest_1samp(v.values, 0.0)
        rows.append([feat, len(v), v.mean(), t.pvalue])

    res = pd.DataFrame(rows, columns=["id","n","log2FoldChange","pvalue"])
    res["padj"] = multipletests(res["pvalue"].values, method="fdr_bh")[1]
    return res.sort_values(["padj","pvalue"])

def split_sets(res_aso: pd.DataFrame, res_pl: pd.DataFrame, id_col: str, fdr=0.05, lfc=0.5):
    a = res_aso[[id_col,"padj","log2FoldChange"]].dropna()
    p = res_pl[[id_col,"padj","log2FoldChange"]].dropna()

    ASO_up = set(a[(a.padj<=fdr) & (a.log2FoldChange>= lfc)][id_col].astype(str))
    ASO_dn = set(a[(a.padj<=fdr) & (a.log2FoldChange<=-lfc)][id_col].astype(str))
    PL_up  = set(p[(p.padj<=fdr) & (p.log2FoldChange>= lfc)][id_col].astype(str))
    PL_dn  = set(p[(p.padj<=fdr) & (p.log2FoldChange<=-lfc)][id_col].astype(str))
    return ASO_up, ASO_dn, PL_up, PL_dn

def plot_two_venns(ASO_up, ASO_dn, PL_up, PL_dn, title, outpath=None):
    if not HAS_VENN:
        print(title)
        print("ASO_up:", len(ASO_up), "PL_up:", len(PL_up), "Overlap:", len(ASO_up & PL_up))
        print("ASO_dn:", len(ASO_dn), "PL_dn:", len(PL_dn), "Overlap:", len(ASO_dn & PL_dn))
        return
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    venn2([ASO_up, PL_up], set_labels=("ASO up","Placebo up"))
    plt.title("UP overlap")
    plt.subplot(1,2,2)
    venn2([ASO_dn, PL_dn], set_labels=("ASO down","Placebo down"))
    plt.title("DOWN overlap")
    plt.suptitle(title)
    plt.tight_layout()
    if outpath:
        plt.savefig(outpath, dpi=200)
    plt.show()

# RNA DE (delta-based)
rna_aso = de_from_deltas_by_arm(meta_rna, delta_rna_all, "ASO").rename(columns={"id":"gene"})
rna_pl  = de_from_deltas_by_arm(meta_rna, delta_rna_all, "placebo").rename(columns={"id":"gene"})
rna_aso.to_csv(f"{OUT_DIR}/rna_delta_DE_ASO.csv", index=False)
rna_pl.to_csv(f"{OUT_DIR}/rna_delta_DE_placebo.csv", index=False)

ASO_up, ASO_dn, PL_up, PL_dn = split_sets(rna_aso, rna_pl, "gene", fdr=FDR_THR, lfc=LFC_THR_RNA)
plot_two_venns(ASO_up, ASO_dn, PL_up, PL_dn,
               f"RNA: ASO vs Placebo (FDR<={FDR_THR}, |log2FC|>={LFC_THR_RNA})",
               outpath=f"{OUT_DIR}/venn_RNA_up_down.png")

# Proteomics DE (delta-based)
prot_aso = de_from_deltas_by_arm(meta_prot, delta_prot_all, "ASO").rename(columns={"id":"protein"})
prot_pl  = de_from_deltas_by_arm(meta_prot, delta_prot_all, "placebo").rename(columns={"id":"protein"})
prot_aso.to_csv(f"{OUT_DIR}/prot_delta_DE_ASO.csv", index=False)
prot_pl.to_csv(f"{OUT_DIR}/prot_delta_DE_placebo.csv", index=False)

ASO_up_p, ASO_dn_p, PL_up_p, PL_dn_p = split_sets(prot_aso, prot_pl, "protein", fdr=FDR_THR, lfc=LFC_THR_PROT)
plot_two_venns(ASO_up_p, ASO_dn_p, PL_up_p, PL_dn_p,
               f"Proteomics: ASO vs Placebo (FDR<={FDR_THR}, |log2FC|>={LFC_THR_PROT})",
               outpath=f"{OUT_DIR}/venn_PROT_up_down.png")

print("Saved DE tables (and Venn figures if available) to outputs/.")
